# Testing LLMs for post-OCR error correction

This notebook uses a large language model (LLM) via [Ollama](https://ollama.com) to correct OCR errors extracted in the previous step.

For information on the test case implemented here in addition to its usefulness for research, see the README in `/code`.


## For working in Colab

## Connect to the GPU:
In the 'Runtime' menu, click on 'Change runtime type' and select 'T4 GPU' under 'Hardware accelerator'. **NOTE:** you will need to be logged in with your Google account to connect your Drive and to use the GPU. Free access is limited; for larger projects, you may need to consider working on an HPC.

## Connect your Google Drive:
Run the code block below to mount your Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Set the path to your working directory

In [2]:
import os

gdrive_path = "/content/drive/My Drive"
wdir_path = os.path.join(gdrive_path, "/THISTLE_project/THISTLE_project")

Working directory set to: /content


## Step 1: Install dependencies
Run this cell first to install all required Python packages.

- **In Google Colab:** packages are installed for your current session and will need reinstalling if you reconnect.
- **In Anaconda (local):** run this once per environment, or install via `requirements.txt` instead (see `code/README.md`).



In [4]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pandas ollama tqdm

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:5 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,389 kB]
Get:12 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:13 https://ppa.launchpadcontent.net

## 2. Imports and Ollama setup
The following code block imports all dependencies and starts the Ollama server.

In [17]:
import pandas as pd
import ollama
from tqdm import tqdm
import time
import subprocess
import threading

def run_ollama_serve():
    subprocess.Popen(['ollama', 'serve'])

if 'colab_env' in locals() and colab_env:
    thread = threading.Thread(target=run_ollama_serve)
    thread.start()
    time.sleep(15)
    print('Ollama server started.')

## 3. Load data and concatenate into one dataframe

This code block imports the ground truth data and OCR output extracted in the previous step.

In [9]:
thistle_df = pd.read_csv('/content/drive/My Drive/THISTLE_project/THISTLE_project/data/ground_truth')
tesseract_df = pd.read_csv('/content/drive/My Drive/THISTLE_project/processed_imgs.csv')


NameError: name 'df' is not defined

In [10]:
thistle_df.head()


,proquest_url,original_ocr,year,manual_transcription,doc_type,png_img
0,http://search.proquest.com/docview/486505058/,ot JolinKerr lut oii St --At DIansc of Gartly ...,1817,"Greenock, or a typhus fever, Mr John Kerr, Tob...",article,1.png
1,http://search.proquest.com/docview/486505009/,THE SC AN EDINBURGH SATURDAY December 27 TOCpK...,1817,"THE SCOTSMAN EDINBURGH, SATURDAY, December 27 ...",masthead,2.png
2,http://search.proquest.com/docview/486501656/,This -iv is published price One Shilling LETTE...,1817,"This day is published, price One Shilling, A L...",classified_ad,3.png
3,http://search.proquest.com/docview/486501551/,CTrr ll TO TllE OUIf-DRV OF BCrNUUIlCU UY -DIl...,1817,"A LETTER TO THE GUILDRY OF EDINBURGH, BY A GUI...",article,4.png
4,http://search.proquest.com/docview/486501454/,The College vo tcd 50 guineas out of the hinds...,1817,The Royal College of Surgeons have voted 50 gu...,article,5.png


In [11]:
tesseract_df.head()

,png_img,tesseract_ocr
0,1.png,"Greenock, of & typhus lever, Wir John: Kerr, “..."
1,10.png,"This day is pyblished, price 26. bound,\noh PR..."
2,100.png,"EDINBURGH ‘':—Printed by CHARLES MAC.\nLAREN, ..."
3,11.png,\n \n \n \n \n \n \n \n \n ...
4,12.png,"tnaller sms to the Infirmary, &c., is uot the ..."


In [13]:
df = thistle_df.merge(tesseract_df, on="png_img", how="left")

In [14]:
df.head()

,proquest_url,original_ocr,year,manual_transcription,doc_type,png_img,tesseract_ocr
0,http://search.proquest.com/docview/486505058/,ot JolinKerr lut oii St --At DIansc of Gartly ...,1817,"Greenock, or a typhus fever, Mr John Kerr, Tob...",article,1.png,"Greenock, of & typhus lever, Wir John: Kerr, “..."
1,http://search.proquest.com/docview/486505009/,THE SC AN EDINBURGH SATURDAY December 27 TOCpK...,1817,"THE SCOTSMAN EDINBURGH, SATURDAY, December 27 ...",masthead,2.png,"THE SCOTSMAN\n\nEDINBURGH, Satunpay, December ..."
2,http://search.proquest.com/docview/486501656/,This -iv is published price One Shilling LETTE...,1817,"This day is published, price One Shilling, A L...",classified_ad,3.png,". This day is‘published, price Oné Shilling,\n..."
3,http://search.proquest.com/docview/486501551/,CTrr ll TO TllE OUIf-DRV OF BCrNUUIlCU UY -DIl...,1817,"A LETTER TO THE GUILDRY OF EDINBURGH, BY A GUI...",article,4.png,"A LETTER TO THE UULLDRY OF EDINKURGH, UY A QUI..."
4,http://search.proquest.com/docview/486501454/,The College vo tcd 50 guineas out of the hinds...,1817,The Royal College of Surgeons have voted 50 gu...,article,5.png,"The Royal College of Surgeons haye voted, 50 g..."


## 4. Define correction function

This code block pulls a model from Ollama, then creates a function that prompts the model to correct the OCR from each row in the dataframe in the 'original_ocr' column and the 'tesseract_ocr' column.

In [18]:
model_name = 'llama3'
ollama.pull(model_name)

def correct_text(text):
    if pd.isna(text) or text.strip() == '':
        return ''

    # prompt: can be altered
    prompt = (
        "Here is OCR-extracted text from a nineteenth-century newspaper in the public domain. "
        "Please correct OCR errors — such as misread characters, broken words, and garbled punctuation — "
        "so that the text makes grammatical and linguistic sense. "
        "Preserve the historical language and spelling conventions of the original; "
        "do not modernise archaic words or add any text that is not present in the original. "
        f"Return only the corrected text, with no additional commentary.\n\n{text}"
    )

    response = ollama.chat(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0.1}
    )
    return response['message']['content']

ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

## 5. Run Correction for All Sources

In [19]:
df['corrected_original'] = [correct_text(t) for t in tqdm(df['original_ocr'])]

df['corrected_tesseract'] = [correct_text(t) for t in tqdm(df['tesseract_text'])]


  0%|          | 0/101 [00:00<?, ?it/s]


NameError: name 'correct_text' is not defined

## 6. Save the results

In [ ]:
df.to_csv('/content/drive/My Drive/THISTLE_project/post_ocr_output.csv', index=False)
